# scRNA-seq 기초 분석 실습
## Part 2: Integration, Clustering & Cell Type Annotation

**건국대학교 이형우 교수님 연구실 온라인 세미나**
Data: GSE210543 (Young: 16PCW, 20PCW | Old: Adult_2, Adult_3)

---

### 학습 목표
1. Young/Old 샘플 배치 효과 보정 (Harmony Integration)
2. PCA → UMAP 차원 축소 및 시각화
3. 최적 클러스터링 Resolution 선택
4. 마커 유전자 기반 세포 타입 어노테이션

> **전제조건**: Part 1 완료 후 저장된 `.rds` 파일 필요

In [ ]:
# 패키지 설치 (처음만)
pkgs <- c("Seurat", "harmony", "dplyr", "ggplot2", "patchwork", "clustree")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE))
    install.packages(p)
}

In [ ]:
library(Seurat)
library(harmony)
library(dplyr)
library(ggplot2)
library(patchwork)

set.seed(42)
cat("Seurat:", as.character(packageVersion("Seurat")), "\n")
cat("harmony:", as.character(packageVersion("harmony")), "\n")

In [ ]:
# Part 1에서 저장한 오브젝트 불러오기
SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
seurat_merged <- readRDS(file.path(SAVE_DIR, "01_seurat_merged_after_QC.rds"))

print(seurat_merged)
table(seurat_merged$group)

---
## Step 7. (Optional) Cell Cycle Scoring

### 🔄 Cell Cycle을 고려해야 하는 이유

세포주기(G1/S/G2M)는 유전자 발현에 강한 영향을 미칩니다.
이를 보정하지 않으면 클러스터가 세포 타입이 아니라 **세포주기 상태**로 나뉠 수 있습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_14.png" width="850"/>

*Fig. 10 — Cell Cycle Scoring: 언제 회귀할지, 언제 남겨둘지*

> **세미나 데이터(망막)** 에서는 발달기 샘플(16PCW, 20PCW)에 증식 세포가 많습니다.
> 세포 타입 구분이 목적이므로 → **Cell Cycle 회귀 적용**

In [ ]:
# Cell cycle 관련 유전자 (Seurat 내장)
s.genes   <- cc.genes$s.genes    # S phase
g2m.genes <- cc.genes$g2m.genes  # G2M phase

# Cell cycle score 계산
seurat_merged <- CellCycleScoring(
  seurat_merged,
  s.features   = s.genes,
  g2m.features = g2m.genes,
  set.ident    = TRUE
)

# 분포 확인
table(seurat_merged$Phase)

---
## Step 8. Scaling & PCA

### 📐 Scaling이 필요한 이유

유전자마다 발현 범위가 다릅니다(어떤 유전자는 0~2, 어떤 유전자는 0~1000).
Scaling은 각 유전자를 평균 0, 분산 1로 맞춰 **동등한 가중치**를 부여합니다.

```
scaled = (expression - mean) / sd
```

Cell cycle 효과를 회귀(regress out)해서 제거합니다.

In [ ]:
# Scaling (cell cycle 회귀 포함)
seurat_merged <- ScaleData(
  seurat_merged,
  vars.to.regress = c("S.Score", "G2M.Score"),  # cell cycle 보정
  features        = rownames(seurat_merged)
)

cat("Scaling complete!\n")

In [ ]:
# PCA 실행
seurat_merged <- RunPCA(
  seurat_merged,
  features = VariableFeatures(seurat_merged),
  npcs     = 50
)

# Elbow Plot — 몇 개의 PC를 사용할지 결정
ElbowPlot(seurat_merged, ndims = 50) +
  ggtitle("Elbow Plot: PC 기여도") +
  geom_vline(xintercept = 30, linetype = "dashed", color = "red") +
  annotate("text", x = 32, y = 3, label = "PC=30 선택", color = "red")

---
## Step 9. Integration (Harmony)

### 🧩 왜 Integration이 필요한가?

서로 다른 샘플(16PCW, 20PCW, Adult_2, Adult_3)은 각각 별도로 sequencing된 데이터입니다.
이 경우 **Batch Effect** — 기술적 변이 — 가 생물학적 차이처럼 보일 수 있습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_15.png" width="850"/>

*Fig. 11 — Integration 방법 비교: Harmony가 세포 타입 구조를 잘 보존 (Tran et al., Genome Biol 2020)*

> **Harmony 선택 이유**: 빠르고 메모리 효율적이며, scRNA-seq에서 검증된 방법
> Seurat v5에서는 `IntegrateLayers()` + `harmony` 사용

In [ ]:
# Seurat v5 방식: Layer-based integration
seurat_merged <- IntegrateLayers(
  object      = seurat_merged,
  method      = HarmonyIntegration,
  orig.reduction = "pca",
  new.reduction  = "harmony",
  group.by.vars  = "sample",  # 배치 변수: 샘플별 보정
  verbose     = FALSE
)

cat("Harmony integration complete!\n")

---
## Step 10. UMAP

### 🗺️ UMAP이란?

UMAP(Uniform Manifold Approximation and Projection)은 고차원(50 PC) 데이터를 **2D로 시각화**하는 방법입니다.
가까운 세포 = 유사한 발현 패턴을 가진 세포

> **주의**: UMAP은 탐색적 시각화 도구입니다. 거리의 절대적 의미는 없습니다.

In [ ]:
# UMAP (Harmony 통합 결과 기반)
seurat_merged <- RunUMAP(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 통합 전/후 비교
p_before <- DimPlot(seurat_merged, reduction = "pca",  group.by = "sample") + ggtitle("Before Integration (PCA)")
p_after  <- DimPlot(seurat_merged, reduction = "umap", group.by = "sample") + ggtitle("After Integration (UMAP)")
p_before + p_after

In [ ]:
# Young vs Old 분포 확인
DimPlot(seurat_merged, reduction = "umap", group.by = "group",
        cols = c("Young" = "#E67E22", "Old" = "#2980B9")) +
  ggtitle("UMAP: Young vs Old") +
  theme_minimal()

---
## Step 11. Clustering

### 🔢 Graph-based Clustering

Seurat은 **KNN 그래프 + Louvain/Leiden 알고리즘**으로 세포를 클러스터링합니다.

**Resolution**: 클러스터 개수를 결정하는 핵심 파라미터
- Resolution ↑ → 클러스터 많아짐 (더 세분화)
- Resolution ↓ → 클러스터 적어짐 (더 뭉침)

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_16.png" width="850"/>

*Fig. 12 — Resolution에 따른 클러스터링 결과 비교 (Res 0.2 ~ 1.2)*

> **Tip**: Resolution 0.4~0.6이 일반적으로 좋은 시작점
> 여러 Resolution 결과를 비교한 후 생물학적으로 의미 있는 것을 선택

In [ ]:
# KNN 그래프 생성
seurat_merged <- FindNeighbors(
  seurat_merged,
  reduction = "harmony",
  dims      = 1:30
)

# 여러 Resolution으로 클러스터링
resolutions <- c(0.2, 0.4, 0.6, 0.8)
for (res in resolutions) {
  seurat_merged <- FindClusters(
    seurat_merged,
    resolution   = res,
    cluster.name = paste0("RNA_snn_res.", res)
  )
  cat(sprintf("Res %.1f: %d clusters\n", res, length(unique(seurat_merged@meta.data[[paste0("RNA_snn_res.", res)]])))  )
}

In [ ]:
# Resolution 비교 시각화
plot_list <- lapply(resolutions, function(res) {
  col_name <- paste0("RNA_snn_res.", res)
  DimPlot(seurat_merged, group.by = col_name, label = TRUE, label.size = 3) +
    ggtitle(paste0("Resolution ", res)) +
    NoLegend()
})

wrap_plots(plot_list, ncol = 2)

In [ ]:
# 최적 Resolution 선택 (세미나: 0.4 사용)
Idents(seurat_merged) <- "RNA_snn_res.0.4"
seurat_merged$seurat_clusters <- Idents(seurat_merged)

DimPlot(seurat_merged, reduction = "umap", label = TRUE, label.size = 4) +
  ggtitle("Final Clustering (Resolution 0.4)") +
  theme_minimal()

---
## Step 12. Marker Gene Identification

### 🔍 FindAllMarkers

각 클러스터를 나머지 전체와 비교하여 **특이적으로 발현되는 유전자**를 찾습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_17.png" width="850"/>

*Fig. 13 — 세포 타입 어노테이션 전략: 자동 어노테이션 vs 수동 어노테이션 (Clake ZA et al., Nat Protoc 2019)*

In [ ]:
# 마커 유전자 탐색 (시간 소요: 5~15분)
# only.pos = TRUE: 해당 클러스터에서 높게 발현되는 유전자만
markers <- FindAllMarkers(
  seurat_merged,
  only.pos          = TRUE,
  min.pct           = 0.25,  # 최소 25% 세포에서 발현
  logfc.threshold   = 0.25   # 최소 log2FC 0.25
)

# Top 5 마커 확인
markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 5) %>%
  print(n = Inf)

In [ ]:
# Top 10 마커 히트맵
top10 <- markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 10)

DoHeatmap(seurat_merged, features = top10$gene) +
  theme(axis.text.y = element_text(size = 6))

---
## Step 13. 세포 타입 어노테이션

### 망막 세포 타입 마커 (GSE210543 기반)

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_18.png" width="850"/>

*Fig. 14 — 세포 타입 어노테이션 방법 및 Tip 4: 연구자의 주관이 중요!*

| 세포 타입 | 마커 유전자 |
|----------|------------|
| Retinal Ganglion Cells (RGC) | RBPMS, SNCG, GAP43 |
| Amacrine cells | GAD1, GAD2, TFAP2A |
| Bipolar cells | VSX2, CABP5, PRKCA |
| Müller glia | RLBP1, GLUL, SLC1A3 |
| Photoreceptors (Rod) | RHO, NR2E3 |
| Photoreceptors (Cone) | ARR3, GNGT2, OPN1LW |
| Horizontal cells | ONECUT2, LHX1 |
| Microglia | CX3CR1, P2RY12 |
| Endothelial cells | PECAM1, CDH5 |
| Progenitor cells | VSX2, FGF19, LIN28B |

> Tip: 발달기(Young) 샘플에는 Progenitor가 많고, Adult(Old)에는 분화된 세포가 많습니다.

In [ ]:
# 알려진 망막 마커 시각화
retinal_markers <- list(
  "Progenitor"   = c("VSX2", "FGF19", "LIN28B"),
  "RGC"          = c("RBPMS", "SNCG"),
  "Amacrine"     = c("GAD1", "TFAP2A"),
  "Bipolar"      = c("CABP5", "PRKCA"),
  "Muller_glia"  = c("RLBP1", "GLUL"),
  "Rod"          = c("RHO", "NR2E3"),
  "Cone"         = c("ARR3", "GNGT2"),
  "Horizontal"   = c("ONECUT2", "LHX1"),
  "Microglia"    = c("CX3CR1", "P2RY12")
)

all_markers <- unlist(retinal_markers)
# 데이터에 존재하는 유전자만 선택
valid_markers <- all_markers[all_markers %in% rownames(seurat_merged)]

DotPlot(seurat_merged, features = valid_markers, group.by = "seurat_clusters") +
  RotatedAxis() +
  ggtitle("Known Retinal Cell Type Markers") +
  theme(axis.text.x = element_text(size = 8))

In [ ]:
# 클러스터 → 세포 타입 매핑 (분석 결과 보고 수정)
# 아래는 예시 - 실제 마커 확인 후 조정 필요
cluster_annotations <- c(
  "0"  = "Muller_glia",
  "1"  = "Progenitor",
  "2"  = "Bipolar",
  "3"  = "RGC",
  "4"  = "Amacrine",
  "5"  = "Rod",
  "6"  = "Cone",
  "7"  = "Horizontal",
  "8"  = "Microglia",
  "9"  = "Unknown"
)

seurat_merged$cell_type <- plyr::mapvalues(
  as.character(seurat_merged$seurat_clusters),
  from = names(cluster_annotations),
  to   = cluster_annotations
)

# 최종 UMAP
DimPlot(seurat_merged, reduction = "umap", group.by = "cell_type",
        label = TRUE, label.size = 3, repel = TRUE) +
  ggtitle("Cell Type Annotation") +
  theme_minimal()

In [ ]:
# Young vs Old: 세포 타입 구성 비교
prop_df <- seurat_merged@meta.data %>%
  group_by(group, cell_type) %>%
  summarise(n = n(), .groups = "drop") %>%
  group_by(group) %>%
  mutate(proportion = n / sum(n))

ggplot(prop_df, aes(x = group, y = proportion, fill = cell_type)) +
  geom_bar(stat = "identity") +
  scale_fill_brewer(palette = "Set3") +
  labs(title = "Cell Type Composition: Young vs Old",
       x = "Group", y = "Proportion") +
  theme_minimal()

In [ ]:
# 최종 저장
saveRDS(seurat_merged, file = file.path(SAVE_DIR, "02_seurat_annotated.rds"))
cat("Saved: 02_seurat_annotated.rds\n")

# 요약
cat("\n=== Final Summary ===\n")
cat("Total cells:", ncol(seurat_merged), "\n")
cat("Cell types:\n")
print(table(seurat_merged$cell_type, seurat_merged$group))

---
## ✅ Part 2 완료!

### 정리

| 단계 | 내용 |
|------|------|
| Cell Cycle | CellCycleScoring → ScaleData 회귀 |
| PCA | RunPCA (50 PC) → ElbowPlot으로 30 PC 선택 |
| Integration | Harmony (샘플 배치 효과 보정) |
| UMAP | RunUMAP (dims = 1:30) |
| Clustering | FindNeighbors + FindClusters (Res 0.4) |
| Marker | FindAllMarkers → DoHeatmap |
| Annotation | Known markers + manual annotation |

### 다음 파트 (Advanced)
**Part 3: CellChat & Monocle3**
→ 세포 간 통신 네트워크 + Pseudotime trajectory 분석

---
*GSE210543 | Human Retina scRNA-seq | KU Online Seminar*